In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

# ==========================================
# [PART 1] CMT 모델 아키텍처 직접 구현 (CVPR 2022)
# ==========================================
class LPU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim, bias=False)
    def forward(self, x):
        return x + self.dwconv(x)

class LMHSA(nn.Module):
    def __init__(self, dim, num_heads, sr_ratio):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.sr_ratio = sr_ratio
        self.q = nn.Linear(dim, dim)
        self.kv = nn.Linear(dim, dim * 2)
        self.proj = nn.Linear(dim, dim)
        if sr_ratio > 1:
            self.sr = nn.Conv2d(dim, dim, kernel_size=sr_ratio, stride=sr_ratio, groups=dim)
            self.norm = nn.LayerNorm(dim)
        else:
            self.sr = None
            self.norm = None

    def forward(self, x, H, W):
        B, N, C = x.shape
        q = self.q(x).reshape(B, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        if self.sr is not None:
            x_2d = x.permute(0, 2, 1).reshape(B, C, H, W)
            x_sr = self.sr(x_2d).reshape(B, C, -1).permute(0, 2, 1)
            x_sr = self.norm(x_sr)
            kv = self.kv(x_sr).reshape(B, -1, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        else:
            kv = self.kv(x).reshape(B, -1, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x

class IRFFN(nn.Module):
    def __init__(self, in_features, hidden_features):
        super().__init__()
        self.conv1 = nn.Conv2d(in_features, hidden_features, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(hidden_features)
        self.act1 = nn.GELU()
        self.dwconv = nn.Conv2d(hidden_features, hidden_features, kernel_size=3, padding=1, groups=hidden_features, bias=False)
        self.bn2 = nn.BatchNorm2d(hidden_features)
        self.act2 = nn.GELU()
        self.conv2 = nn.Conv2d(hidden_features, in_features, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(in_features)

    def forward(self, x):
        residual = x
        x = self.act1(self.bn1(self.conv1(x)))
        x = self.act2(self.bn2(self.dwconv(x)))
        x = self.bn3(self.conv2(x))
        return x + residual

class CMTBlock(nn.Module):
    def __init__(self, dim, num_heads, sr_ratio, expansion_ratio=4):
        super().__init__()
        self.lpu = LPU(dim)
        self.ln1 = nn.LayerNorm(dim)
        self.lmhsa = LMHSA(dim, num_heads, sr_ratio)
        self.ln2 = nn.LayerNorm(dim)
        self.irffn = IRFFN(dim, int(dim * expansion_ratio))

    def forward(self, x):
        B, C, H, W = x.shape
        x = self.lpu(x)
        x_flat = x.flatten(2).transpose(1, 2)
        x_norm1 = self.ln1(x_flat)
        attn_out = self.lmhsa(x_norm1, H, W)
        x = x + attn_out.transpose(1, 2).reshape(B, C, H, W)
        x_norm2 = self.ln2(x.flatten(2).transpose(1, 2)).transpose(1, 2).reshape(B, C, H, W)
        x = self.irffn(x_norm2)
        return x

class PatchEmbed(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        x = self.proj(x)
        B, C, H, W = x.shape
        x_flat = x.flatten(2).transpose(1, 2)
        x_norm = self.norm(x_flat)
        return x_norm.transpose(1, 2).reshape(B, C, H, W)

class CMT_S(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU()
        )
        self.patch_embed1 = PatchEmbed(32, 64)
        self.stage1 = nn.Sequential(*[CMTBlock(64, num_heads=1, sr_ratio=8) for _ in range(3)])
        self.patch_embed2 = PatchEmbed(64, 128)
        self.stage2 = nn.Sequential(*[CMTBlock(128, num_heads=2, sr_ratio=4) for _ in range(3)])
        self.patch_embed3 = PatchEmbed(128, 256)
        self.stage3 = nn.Sequential(*[CMTBlock(256, num_heads=4, sr_ratio=2) for _ in range(16)])
        self.patch_embed4 = PatchEmbed(256, 512)
        self.stage4 = nn.Sequential(*[CMTBlock(512, num_heads=8, sr_ratio=1) for _ in range(3)])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.proj = nn.Conv2d(512, 1280, kernel_size=1)
        self.head = nn.Linear(1280, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(self.patch_embed1(x))
        x = self.stage2(self.patch_embed2(x))
        x = self.stage3(self.patch_embed3(x))
        x = self.stage4(self.patch_embed4(x))
        x = self.avgpool(x)
        x = self.proj(x)
        x = x.view(x.size(0), -1)
        x = self.head(x)
        return x

# ==========================================
# [PART 2] 훈련 함수 정의 (AMP 혼합정밀 적용)
# ==========================================
def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs, save_path):
    best_acc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    # ✅ AMP: GradScaler 생성 (CUDA일 때만 실제로 활성화)
    use_amp = (device.type == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    if use_amp:
        print("\n🔥 본격적인 CMT 훈련을 시작합니다! (AMP 혼합정밀 ON)")
    else:
        print("\n🔥 본격적인 CMT 훈련을 시작합니다! (CPU 모드)")

    for epoch in range(epochs):
        start_time = time.time()

        # 훈련
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
        for inputs, labels in train_loop:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            # ✅ AMP: autocast 영역 안에서 forward + loss 계산
            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            # ✅ AMP: scaler를 통한 backward / step / update
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
            train_loop.set_postfix(loss=f"{loss.item():.4f}")  # 배치마다 현재 loss 표시

        epoch_train_loss = train_loss / train_total
        epoch_train_acc = train_correct / train_total

        # 검증
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)
        with torch.no_grad():
            for inputs, labels in val_loop:
                inputs, labels = inputs.to(device), labels.to(device)
                # ✅ AMP: 검증에서도 autocast 사용 (속도 향상)
                with torch.cuda.amp.autocast(enabled=use_amp):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total

        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)

        elapsed_time = time.time() - start_time
        print(f"Epoch {epoch+1:02d}/{epochs:02d} | Time: {elapsed_time:.0f}s | "
              f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")

        if epoch_val_acc > best_acc:
            best_acc = epoch_val_acc
            torch.save(model.state_dict(), save_path)
            print(f"  🌟 최고 성능 갱신! 모델 저장됨: {save_path}")

    print(f"\n🎉 훈련 종료! 최고 검증 정확도: {best_acc:.4f}")
    return history

def plot_history(history, save_dir):
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('CMT Model Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('CMT Model Loss')
    plt.legend()

    graph_path = os.path.join(save_dir, "training_log.png")
    plt.savefig(graph_path)
    print(f"📊 학습 그래프가 저장되었습니다: {graph_path}")
    plt.show()

# 채널 수가 다른 이미지(흑백/RGBA)가 섞여 있어도 안전하게 3채널 RGB로 강제 변환
def rgb_loader(path):
    return Image.open(path).convert("RGB")

# ==========================================
# [PART 3] 메인 실행부 (Windows 멀티프로세싱 에러 완벽 차단)
# ==========================================
if __name__ == "__main__":
    # 🚨 이 아래에 있는 코드들은 오직 '메인 관리자'만 실행합니다! (프리징 완벽 해결)
    PATH_LOCAL = r"C:\Users\user\Desktop\졸작_최종_파이프라인"
    PATH_ONEDRIVE = r"C:\Users\user\OneDrive\바탕 화면\졸작_최종_파이프라인"

    if os.path.exists(PATH_LOCAL):
        DATA_DIR = PATH_LOCAL
    elif os.path.exists(PATH_ONEDRIVE):
        DATA_DIR = PATH_ONEDRIVE
    else:
        print("❌ 데이터 파이프라인 폴더를 찾을 수 없습니다. 경로를 확인해주세요.")
        exit()

    MODEL_SAVE_PATH = os.path.join(DATA_DIR, "best_cmt_model.pt")

    BATCH_SIZE = 32
    EPOCHS = 30
    LEARNING_RATE = 1e-4

    # 여전히 메모리 문제로 멈춘다면 NUM_WORKERS를 0으로 두세요.
    NUM_WORKERS = 0

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 학습 장치: {device} 로 구동됩니다.")

    # ✅ 핵심 수정: Resize 추가 (이미지 크기를 224x224로 통일해야 배치 생성이 가능)
    common_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print("📂 데이터셋을 불러오는 중입니다...")
    train_dataset = datasets.ImageFolder(
        root=os.path.join(DATA_DIR, 'train'),
        transform=common_transform,
        loader=rgb_loader
    )
    val_dataset = datasets.ImageFolder(
        root=os.path.join(DATA_DIR, 'val'),
        transform=common_transform,
        loader=rgb_loader
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    NUM_CLASSES = len(train_dataset.classes)
    print(f"✅ 총 {NUM_CLASSES}개의 의류 클래스를 감지했습니다.")

    print("🧠 자체 구현한 오리지널 CMT-S 모델을 로딩합니다...")
    model = CMT_S(num_classes=NUM_CLASSES).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

    # 훈련 함수 호출
    history = train_model(model, train_loader, val_loader, criterion, optimizer, device, EPOCHS, MODEL_SAVE_PATH)

    # 결과 그래프 그리기
    plot_history(history, DATA_DIR)

🚀 학습 장치: cuda 로 구동됩니다.
📂 데이터셋을 불러오는 중입니다...


C:\Users\user\AppData\Local\Temp\ipykernel_28668\3536816860.py:149: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


✅ 총 22개의 의류 클래스를 감지했습니다.
🧠 자체 구현한 오리지널 CMT-S 모델을 로딩합니다...

🔥 본격적인 CMT 훈련을 시작합니다! (AMP 혼합정밀 ON)


Epoch 1/30 [Train]:   0%|          | 0/3438 [00:00<?, ?it/s]C:\Users\user\AppData\Local\Temp\ipykernel_28668\3536816860.py:168: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
